# ExaRanker (2023)
[[paper]](https://arxiv.org/pdf/2305.03531)<br>
ExaRanker = Explanation-augmented Neural Ranker

__ExaRanker__ — это метод обучения Cross-Encoder моделей для переранжирования (Re-ranking), который использует объяснения, сгенерированные большими языковыми моделями (LLM), в качестве дополнительных данных для обучения. Подход позволяет значительно повысить качество поиска в условиях ограниченного объема размеченных данных (Few-shot learning).

__Постановка задачи__<br>
В классическом пайплайне информационного поиска после первого этапа (Retrieval) следует этап Re-ranking. Задача переранжировщика — максимально точно упорядочить небольшое подмножество документов (например, топ-100), полученных на первом шаге, используя более тяжелые и точные модели.

__Мотивация__<br>
Современные Cross-Encoder модели (например, на базе BERT или T5) обучаются на парах "запрос-документ" с бинарными метками релевантности (0 или 1). Проблема в том, что такие метки не несут информации о том, *почему* документ релевантен. Большие языковые модели (GPT-3.5, GPT-4) обладают отличными способностями к рассуждению, но они слишком дороги и медленны для прямого использования в Re-ranking на больших потоках запросов. Авторы ExaRanker задались вопросом: можно ли перенести "знание о причинах релевантности" из LLM в компактную модель-ранжировщик.

__Существующие подходы__<br>
На момент появления ExaRanker (2023) основными методами были:
- MonoBERT / MonoT5 (2019/2020): стандартные Cross-Encoder модели, обучаемые только на классификацию релевантности. Они требуют десятков тысяч обучающих примеров для достижения высокой точности.
- InPars (2022): метод синтетической генерации данных, где LLM генерирует запросы к существующим документам. Это расширяет датасет, но обучение все равно идет на простых метках без объяснений.
- PROMPT-based Re-ranking (2023): прямое использование LLM для ранжирования через Few-shot промпты. Это дает высокую точность, но крайне неэффективно по времени и стоимости (Inference Latency).

__Идея__<br>
Вместо того чтобы учить модель просто предсказывать число (score), авторы предложили обучать её генерировать объяснение релевантности, за которым следует сама метка. Это заставляет модель в процессе обучения выстраивать логические связи между текстом запроса и документа, используя LLM как "умного учителя", который размечает не только ответ, но и ход решения.

<img src="img/exaranker/exaranker1.png" width=500>

__Архитектура__<br>
ExaRanker не меняет базовую архитектуру трансформера, а модифицирует формат входных и выходных данных.
1. Teacher Model: мощная LLM (например, GPT-3.5), которая генерирует объяснения.
2. Student Model: более компактная Encoder-Decoder модель (например, T5-base или T5-large).
3. Формат данных: на вход подается пара `Query + Document`, а целевой последовательностью (target) становится строка вида `Explanation + Relevance Label`

<img src="img/exaranker/exaranker2.png" width=500>

__Алгоритм обучения__<br>
Процесс обучения состоит из трех этапов:
1. Explanation Generation: Для небольшого набора данных (Few-shot) с известными метками релевантности вызывается LLM. Ей подается промпт: "Объясни, почему этот документ релевантен/нерелевантен данному запросу". Полученные тексты (Explanations) сохраняются.
2. Data Augmentation: Формируется обучающая выборка, где для каждой пары `(q, d)` целевым значением является сгенерированное объяснение, дополненное токеном релевантности (например, "true" или "false").
3. Multi-task Fine-tuning: Модель T5 обучается минимизировать Negative Log-Likelihood при генерации всей последовательности. Благодаря механизму Self-attention модель учится фокусироваться на тех частях документа, которые LLM выделила в объяснении.

__Алгоритм инференса__<br>
Во время работы (Inference) ExaRanker может работать в двух режимах:
1. Full Generation: Модель генерирует и объяснение, и метку. Это полезно для интерпретируемого поиска, но медленнее.
2. Label-only Scoring: Поскольку нас интересует только ранжирование, мы можем смотреть на вероятность (Logit) генерации токенов "true"/"false" на определенных позициях, не генерируя весь текст объяснения. Это позволяет сохранить скорость стандартного MonoT5 при повышенной точности за счет того, что веса модели "пропитаны" логикой объяснений после обучения.

__Результаты__<br>
Эксперименты проводились на датасетах MS MARCO и TREC DL:
- В сценарии Few-shot (всего 100 примеров для обучения) ExaRanker показал прирост метрики nDCG@10 на 12-15пп по сравнению со стандартным MonoT5, обученным на том же количестве данных.
- Модель ExaRanker, обученная на 100 примерах с объяснениями, достигает качества, сопоставимого с моделями, обученными на 2000–5000 примеров с обычными метками.
- При увеличении размера Student модели до T5-3B преимущество метода сохраняется, что доказывает эффективность дистилляции именно логики рассуждений, а не просто статистических паттернов.

__References:__<br>
- [github](https://github.com/unicamp-dl/ExaRanker?tab=readme-ov-file)

## 📝 Критический анализ

```markdown
# ExaRanker (2023)
---
[[paper]](https://arxiv.org/pdf/2305.03531)<br>
ExaRanker = Explanation-augmented Neural Ranker

ExaRanker — метод обучения Cross-Encoder моделей для Re-ranking, использующий объяснения, сгенерированные большими языковыми моделями (LLM), для улучшения качества поиска в условиях Few-shot learning.

__Задача__<br>
После Retrieval этапа в информационном поиске следует Re-ranking, где задача — точно упорядочить топ-100 документов, используя более точные модели.

__Мотивация__<br>
Cross-Encoder модели (например, BERT, T5) обучаются на парах "запрос-документ" с бинарными метками, не объясняющими релевантность. LLM, такие как GPT-3.5, обладают отличными способностями к рассуждению, но слишком дороги для прямого использования. ExaRanker переносит "знание о причинах релевантности" из LLM в компактную модель-ранжировщик.

__Существующие подходы__<br>
- MonoBERT / MonoT5 (2019/2020): требуют много данных для обучения.
- InPars (2022): расширяет датасет синтетически, но без объяснений.
- PROMPT-based Re-ranking (2023): использует LLM, но неэффективно по времени и стоимости.

__Идея__<br>
Обучать модель генерировать объяснение релевантности, за которым следует метка, используя LLM как "умного учителя".

__Архитектура__<br>
ExaRanker модифицирует формат данных:
1. Teacher Model: LLM (например, GPT-3.5) генерирует объяснения.
2. Student Model: компактная Encoder-Decoder модель (например, T5-base).
3. Формат данных: `Query + Document` -> `Explanation + Relevance Label`.

<img src="img/img.png" width=500>

__Алгоритм обучения__<br>
1. Explanation Generation: LLM генерирует объяснения для небольшого набора данных.
2. Data Augmentation: создается выборка с объяснениями и метками.
3. Multi-task Fine-tuning: T5 обучается минимизировать Negative Log-Likelihood, фокусируясь на частях документа, выделенных в объяснении.

__Алгоритм инференса__<br>
1. Full Generation: генерирует объяснение и метку.
2. Label-only Scoring: оценивает вероятность генерации "true"/"false", сохраняя скорость MonoT5.

__Результаты__<br>
На MS MARCO и TREC DL ExaRanker в Few-shot сценарии увеличил nDCG@10 на 12-15пп по сравнению с MonoT5. Модель, обученная на 100 примерах с объяснениями, достигает качества, сопоставимого с моделями, обученными на 2000–5000 примеров с обычными метками.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций ExaRanker на Python

# Импортируем необходимые библиотеки
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Инициализация модели и токенизатора T5
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base")

# Пример данных: запрос и документ
query = "What is the capital of France?"
document = "Paris is the capital city of France, known for its art, fashion, and culture."

# Генерация объяснения с использованием Teacher Model (LLM)
# В реальном сценарии это будет вызов к LLM, например, GPT-3.5
def generate_explanation(query, document):
    # Пример объяснения, которое могло бы быть сгенерировано LLM
    explanation = "The document states that Paris is the capital city of France, which directly answers the query."
    return explanation

# Генерация объяснения
explanation = generate_explanation(query, document)

# Формирование обучающей выборки
# Входные данные: Query + Document
input_text = f"Query: {query} Document: {document}"

# Целевая последовательность: Explanation + Relevance Label
target_text = f"{explanation} true"

# Токенизация входных и целевых данных
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
labels = tokenizer(target_text, return_tensors="pt").input_ids

# Обучение модели
# В реальном сценарии здесь будет цикл обучения с оптимизацией
outputs = model(input_ids=input_ids, labels=labels)
loss = outputs.loss
print(f"Training Loss: {loss.item()}")

# Инференс: Label-only Scoring
# Мы можем использовать модель для оценки вероятности релевантности
def label_only_scoring(query, document):
    input_text = f"Query: {query} Document: {document}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids

    # Генерация только метки релевантности
    output_sequences = model.generate(input_ids, max_length=10, num_return_sequences=1)
    decoded_output = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
    
    # Извлечение метки релевантности
    relevance_label = decoded_output.split()[-1]
    return relevance_label

# Пример использования модели для предсказания релевантности
predicted_label = label_only_scoring(query, document)
print(f"Predicted Relevance Label: {predicted_label}")

# В этом примере мы показали, как ExaRanker использует объяснения для улучшения обучения модели.
# Мы также продемонстрировали, как можно использовать модель для быстрого предсказания метки релевантности.
```

Этот код иллюстрирует основные концепции ExaRanker, включая генерацию объяснений и использование их для обучения модели. Мы также показали, как можно использовать модель для быстрого предсказания метки релевантности, что является одной из ключевых особенностей ExaRanker.